## Introduction

- Finding the right talent for technology companies is a challenging and resource-intensive task. Recruiters must not only understand the client’s requirements but also identify what makes a candidate stand out and where to locate such individuals. This manual process consumes significant time and effort, often slowing down the hiring pipeline.

- To streamline this process, we are developing a machine learning–powered solution that automates candidate evaluation and ranking. By analyzing role-specific keywords such as “full-stack software engineer” or “engineering manager”, our system generates a ranked list of potential candidates. Recruiters can then review these lists and provide feedback by highlighting the candidate they believe is the best fit. Each time feedback is given, the system learns and re-ranks candidates accordingly, ensuring that future recommendations align more closely with real hiring decisions.

- This solution not only saves time but also enhances accuracy, helping recruiters quickly identify top talent while maintaining flexibility to adapt to client preferences. Ultimately, it represents a smarter, more efficient way to connect technology companies with the individuals best suited to their needs.


## Data Description

The dataset used in this project originates from our candidate sourcing activities. To ensure privacy, all personally identifiable information has been removed, and each candidate is instead represented by a unique numeric identifier.
The dataset includes the following attributes:

- id: A numeric identifier assigned to each candidate.
- job_title: The candidate’s professional title.
- location: The geographical location of the candidate.
- connections: The number of professional connections the candidate has. Values such as “500+” indicate more than 500 connections.

The target variable we aim to predict is:

- fit: A numeric score between 0 and 1 representing how well a candidate matches the requirements of a given role.
For this project, we focus on roles related to human resources, using keywords such as “Aspiring human resources” or “seeking human resources” to guide candidate selection and evaluation.


## Project Goals

- The primary objective of this project is to predict how well a candidate fits a given role based on the information available in the dataset. Specifically, we aim to model the target variable fit, which represents the probability (between 0 and 1) that a candidate is suitable for the role.

- By leveraging attributes such as job title, location, and professional connections, the system will learn patterns that distinguish strong candidates from weaker matches. This predictive capability will allow us to move beyond manual keyword searches (e.g., “Aspiring human resources” or “seeking human resources”) and instead generate ranked candidate lists that reflect the likelihood of success in a role.

- Ultimately, the goal is to build a machine learning pipeline that not only automates candidate evaluation but also adapts to recruiter feedback, ensuring that the recommendations align with real-world hiring decisions. This approach will save time, improve accuracy, and provide a scalable solution for talent sourcing in technology-driven industries.


## Table of Contents

1. Libraries
2. Data Loading
3. Data preprocessing and text cleaning


1. Libraries

In [38]:
# basics
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
# from sklearn import metrics
# from sklearn.model_selection import train_test_split


import nltk 
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer 
from nltk.tokenize import word_tokenize


# Modeling


import warnings
warnings.filterwarnings("ignore")

In [2]:
import os
import random
# Define a fixed random state value
RANDOM_STATE =4230
# Set Python's built-in random seed
random.seed(RANDOM_STATE)
# Set PYTHONHASHSEED environment variable for reproducibility in some cases
os.environ['PYTHONHASHSEED'] = str(RANDOM_STATE)

# Set NumPy's random seed
np.random.seed(RANDOM_STATE)

### 2. Data Loading

In [19]:
Data_human=pd.read_csv("Potential Talents.csv") 
Data_human.head(15)

,id,job_title,location,connection,fit
0,1,2019 C.T. Bauer College of Business Graduate (...,"Houston, Texas",85,NaN
1,2,Native English Teacher at EPIK (English Progra...,Kanada,500+,NaN
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,NaN
3,4,People Development Coordinator at Ryan,"Denton, Texas",500+,NaN
4,5,Advisory Board Member at Celal Bayar University,"İzmir, Türkiye",500+,NaN
5,6,Aspiring Human Resources Specialist,Greater New York City Area,1,NaN
6,7,Student at Humber College and Aspiring Human R...,Kanada,61,NaN
7,8,HR Senior Specialist,San Francisco Bay Area,500+,NaN
8,9,Student at Humber College and Aspiring Human R...,Kanada,61,NaN
9,10,Seeking Human Resources HRIS and Generalist Po...,Greater Philadelphia Area,500+,NaN



### 3. Data preprocessing and text cleaning

In [20]:
Data_human.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104 entries, 0 to 103
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          104 non-null    int64  
 1   job_title   104 non-null    object 
 2   location    104 non-null    object 
 3   connection  104 non-null    object 
 4   fit         0 non-null      float64
dtypes: float64(1), int64(1), object(3)
memory usage: 4.2+ KB


- Rows and columns: 104 Rows, 05 columns
- Types of variables: 01 Numeric variables, 03 Categorical variables and 01 float variable 

In [21]:
Data_human.isnull().sum()

id              0
job_title       0
location        0
connection      0
fit           104
dtype: int64

In Human Resources dataset, rows with duplicate titles may not always share identical wording; instead, they can convey the same meaning through different phrasing. This represents semantic duplication rather than exact textual repetition. 


In [22]:
#count duplicated rows in the total dataset
Data_human.loc[Data_human.duplicated(), :]
print(Data_human.duplicated().sum())

0


In [26]:
Data_human.job_title.value_counts()

job_title
2019 C.T. Bauer College of Business Graduate (Magna Cum Laude) and aspiring Human Resources professional                 7
Aspiring Human Resources Professional                                                                                    7
Student at Humber College and Aspiring Human Resources Generalist                                                        7
People Development Coordinator at Ryan                                                                                   6
Native English Teacher at EPIK (English Program in Korea)                                                                5
Aspiring Human Resources Specialist                                                                                      5
HR Senior Specialist                                                                                                     5
Advisory Board Member at Celal Bayar University                                                                          4
Seekin

Before applying any advanced Natural Language Processing (NLP) techniques, it’s essential to clean and normalize raw text data. Real-world text often contains inconsistencies, noise, and irrelevant elements that can negatively impact model performance. Text cleaning ensures that the input is standardized, meaningful, and ready for analysis.

In this project, we will follow a structured pipeline using NLTK and Python’s built-in libraries to prepare our text:

- Case Normalization: Convert all text to lowercase for consistency.
- Tokenization: Split text into words or sentences using NLTK’s tokenizers.
- Removing Noise: Eliminate punctuation, numbers, special characters, HTML tags, URLs, and extra whitespace.
- Stopword Removal: Filter out common words (like the, is, a) that add little semantic value.
- Normalization (Stemming/Lemmatization): Reduce words to their root form to unify variations (e.g., running → run).

These steps transform messy, unstructured text into a clean and analyzable format, forming the foundation for downstream tasks such as feature extraction, sentiment analysis, or building machine learning models.


In [ ]:
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

In [41]:
def clean_text(text):
	"""Clean and normalize text by converting to lowercase and removing special characters."""
	text = str(text).lower()
	text = re.sub(r'[^a-z0-9\s]', '', text)  # Remove special characters
	text = re.sub(r'\s+', ' ', text).strip()  # Remove extra whitespace
	text = re.sub(r'\d', ' ', text)  # Remove digits
	text = re.sub(r'\s\s+', ' ', text)  # Remove extra spaces
	# Tokenize
	word_tokenized_text = word_tokenize(text)
	
	# Remove stop words
	stop_words = set(stopwords.words('english'))
	word_tokenized_text = [word for word in word_tokenized_text if word not in stop_words]
	# Lemmatize
	lemmatizer = WordNetLemmatizer()
	word_tokenized_text = [lemmatizer.lemmatize(word) for word in word_tokenized_text]
	# Join tokens back into a string
	text = ' '.join(word_tokenized_text)	
	return text


clean_title = Data_human['job_title'].apply(clean_text)
clean_title.head(15)

0     ct bauer college business graduate magna cum l...
1     native english teacher epik english program korea
2                  aspiring human resource professional
3                   people development coordinator ryan
4          advisory board member celal bayar university
5                    aspiring human resource specialist
6     student humber college aspiring human resource...
7                                  hr senior specialist
8     student humber college aspiring human resource...
9       seeking human resource hris generalist position
10                           student chapman university
11    svp chro marketing communication csr officer e...
12    human resource coordinator intercontinental bu...
13    ct bauer college business graduate magna cum l...
14    ct bauer college business graduate magna cum l...
Name: job_title, dtype: object